In [ ]:
# 1. Create and activate virtual environment
python -m venv nlp_env
# On Windows: nlp_env\Scripts\activate
# On macOS/Linux: source nlp_env/bin/activate

# 2. Install required libraries
pip install requests nltk spacy wordcloud matplotlib scikit-learn

# 3. Download spaCy language model and NLTK data corpora
python -m spacy download en_core_web_sm
python -c "import nltk; nltk.download('punkt'); nltk.download('stopwords'); nltk.download('averaged_perceptron_tagger'); nltk.download('maxent_ne_chunker'); nltk.download('words')"

In [ ]:
import re
import requests
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import spacy

urls = [
    "https://www.gutenberg.org/files/11/11-0.txt",
    "https://www.gutenberg.org/files/12/12-0.txt",
    "https://www.gutenberg.org/files/29042/29042-0.txt"
]

def load_texts(url_list):
    corpus = []
    for url in url_list:
        response = requests.get(url)
        response.encoding = 'utf-8'
        text = response.text

        # Slice out Project Gutenberg headers and footers
        start_match = re.search(r'\*\*\* START OF THIS PROJECT GUTENBERG EBOOK.*?\*\*\*', text, re.IGNORECASE)
        end_match = re.search(r'\*\*\* END OF THIS PROJECT GUTENBERG EBOOK.*', text, re.IGNORECASE)

        if start_match:
            text = text[start_match.end():]
        if end_match:
            text = text[:end_match.start()]

        # Optional: Clean non-word characters if needed (preserving punctuation/letters)
        # For standard NLP tokenization, we keep letters and basic punctuation
        corpus.append(text)
    return corpus

corpus = load_texts(urls)

# Print first 200 characters of each cleaned text
for i, text in enumerate(corpus, 1):
    print(f"--- Book {i} (First 200 chars) ---")
    print(text[:200].strip())
    print("\n")

In [ ]:
tokenized_books = [word_tokenize(book.lower()) for book in corpus]

for i, tokens in enumerate(tokenized_books, 1):
    print(f"--- Book {i}: First 150 tokens ---")
    print(tokens[:150])
    print("\n")

In [ ]:
stop_words = set(stopwords.words('english'))
filtered_books = [[word for word in tokens if word.isalnum() and word not in stop_words] for tokens in tokenized_books]

# Verify removal with count()
test_words = ['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves']
for i, (orig, filtered) in enumerate(zip(tokenized_books, filtered_books), 1):
    print(f"--- Book {i} Stopword Check ---")
    for tw in test_words:
        print(f"'{tw}': Before = {orig.count(tw)}, After = {filtered.count(tw)}")
    print("\n")

In [ ]:
stemmer = PorterStemmer()
stemmed_books = [[stemmer.stem(word) for word in tokens] for tokens in filtered_books]

for i, stemmed in enumerate(stemmed_books, 1):
    print(f"--- Book {i}: First 50 Stemmed Tokens ---")
    print(stemmed[:50])
    print("\n")

In [ ]:
nlp = spacy.load("en_core_web_sm")

lemmatized_books = []
for book_text in corpus:
    doc = nlp(book_text)
    lemmas = [token.lemma_.lower() for token in doc if token.is_alpha and token.lemma_.lower() not in stop_words]
    lemmatized_books.append(lemmas)

for i, lemmas in enumerate(lemmatized_books, 1):
    print(f"--- Book {i}: First 50 Lemmatized Tokens ---")
    print(lemmas[:50])
    print("\n")

In [ ]:
pos_tagged_books = [nltk.pos_tag(tokens[:1000]) for tokens in filtered_books] # Tagging a sample for performance
for i, tags in enumerate(pos_tagged_books, 1):
    print(f"--- Book {i}: First 10 POS Tags ---")
    print(tags[:10])
    print("\n")

In [ ]:
ner_books = [nltk.ne_chunk(pos_tag) for pos_tag in pos_tagged_books]
for i, tree in enumerate(ner_books, 1):
    entities = [chunk for chunk in tree if hasattr(chunk, 'label')]
    print(f"--- Book {i}: Sample Entities Found ---")
    print(entities[:10])
    print("\n")

In [ ]:
import matplotlib.pyplot as plt
from wordcloud import WordCloud

for i, book_text in enumerate(corpus, 1):
    wc = WordCloud(width=800, height=400, background_color='white', stopwords=stop_words).generate(book_text)

    plt.figure(figsize=(10, 5))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.title(f"Word Cloud - Book {i}")
    plt.show()

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

# Join lemmatized tokens back into strings for CountVectorizer
lemmatized_corpus_strs = [" ".join(book) for book in lemmatized_books]

vectorizer = CountVectorizer(max_features=5)
X_bow = vectorizer.fit_transform(lemmatized_corpus_strs)

print("Feature Names (Top 5 words per corpus overall):", vectorizer.get_feature_names_out())
print("BoW Matrix Shape:", X_bow.shape)
print("BoW Array Representation:\n", X_bow.toarray())

In [ ]:
for i, book_lemmas in enumerate(lemmatized_books, 1):
    freq_dist = nltk.FreqDist(book_lemmas)
    top_5 = freq_dist.most_common(5)
    words, counts = zip(*top_5)

    plt.figure(figsize=(6, 6))
    plt.pie(counts, labels=words, autopct='%1.1f%%', startangle=140)
    plt.title(f"Top 5 Frequent Words in Book {i}")
    plt.show()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Using min_df=1, max_df=2 as specified for small corpora
tfidf_vectorizer = TfidfVectorizer(min_df=1, max_df=2)
X_tfidf = tfidf_vectorizer.fit_transform(lemmatized_corpus_strs)
feature_names = tfidf_vectorizer.get_feature_names_out()

df_tfidf = pd.DataFrame(X_tfidf.toarray(), columns=feature_names)

for i in range(len(corpus)):
    top_docs = df_tfidf.iloc[i].nlargest(5)
    words = top_docs.index.tolist()
    scores = top_docs.values.tolist()

    plt.figure(figsize=(6, 6))
    plt.pie(scores, labels=words, autopct='%1.1f%%', startangle=140)
    plt.title(f"Top 5 TF-IDF Relevant Words - Book {i+1}")
    plt.show()